In [2]:
import ROOT

import numpy as np
import glob
import os
import re
import uuid
import json
import natsort


In [83]:
def file_load(base_path):
    hist_index = {}
    all_angles=[]
    #extracts the uniqe angles from a folder with ange-position files

    for angle_dir in os.listdir(base_path):
        angle_path = os.path.join(base_path, angle_dir)
        if not os.path.isdir(angle_path):
            continue
        matchang = re.search(r'(Angle[\d.]+)', angle_path)
        angle1 = matchang.group(1)
        all_angles.append(angle1)
    unique_angle = list(set(all_angles))
    unique_angle=natsort.natsorted(unique_angle)

    # takes uniqe angles and makes a directory with the tree angle-position
    for angle in unique_angle:
        hist_index[angle] = {}
        for angle_dir in os.listdir(base_path):
            angle_path = os.path.join(base_path, angle_dir)
            if not os.path.isdir(angle_path):
                continue
            matchang = re.search(r'(Angle[\d.]+)', angle_path)
            angleset = matchang.group(1)  
            if angle != angleset:
                continue
            matchpos = re.search(r'(Pos[\d.]+)', angle_path)
            if matchpos is None:
                print(f"Could not find position in filename: {angle_path}")
                continue
            position = matchpos.group(1)
            hist_index[angle][position] = {}
            #takes the folders that are now sorted and takes basic and the info jason
            for file in os.listdir(angle_path):
                file_path = os.path.join(angle_path, file)
                if re.search(r'basic_', file)is not None and file.endswith(".root"):
                    open_file= ROOT.TFile.Open(file_path)

                    for key in open_file.GetListOfKeys():
                        obj_name = key.GetName()
                        obj_class = key.GetClassName()
                        obj_cycle = key.GetCycle()
                        if int(re.findall(r"-?\d+\.?\d*", obj_name)[0])<50:
                            hist_index[angle][position][obj_name] = {
                                "filepath": file_path,
                                "class": obj_class,
                                "cycle": obj_cycle
                }
                    open_file.Close()
                elif re.search(r'EventProperties', file)is not None and file.endswith(".json"):
                    hist_index[angle][position][file] = {
                        "filepath": file_path,
                    }
                else:
                    continue





    return(hist_index)
def save_index(index,position):
    with open(position, "w") as f:
        json.dump(index, f, indent=4)


In [84]:
base_path1="Band_class_training data/TE_011/Results"
index1=file_load(base_path1)
if __name__ == "__main__":
    output_file = os.path.join(base_path1, "Te011.json")
    save_index(index1,output_file)


In [ ]:
base_path2="Band_class_training data/TE_012/Results"
index2=file_load(base_path2)
if __name__ == "__main__":
    output_file = os.path.join(base_path2, "Te012.json")
    save_index(index2,output_file)